# Inferencja modelu Qwen3.5 do klasyfikacji utworów rapowych


## Import bibliotek


In [1]:
import os

import numpy as np
import pandas as pd
import torch

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

# Unsloth przed transformers (optymalizacje i patch Qwen3.5)
from unsloth import FastModel

from peft import PeftModel
from safetensors import safe_open
from transformers import AutoModelForSequenceClassification, AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0615 00:29:51.611000 27276 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


[ERROR] `loss` is part of Qwen3VLMoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_vl_moe\modeling_qwen3_vl_moe.py.
[ERROR] `logits` is part of Qwen3VLMoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_vl_moe\modeling_qwen3_vl_moe.py.
[ERROR] `loss` is part of Qwen3_5MoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\transformers\models\qwen3_5_moe\modeling_qwen3_5_moe.py.
[ERROR] `logits` is part of Qwen3_5MoeCausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in c:\Users\Damia\D

## Wczytanie danych i etykiet


In [2]:
DATASET_DIR = "dataset"

# Etykiety odtwarzamy z train.json, a zwrotki do inferencji bierzemy ze zbioru testowego.
train_df = pd.read_json(os.path.join(DATASET_DIR, "train.json"))
test_df = pd.read_json(os.path.join(DATASET_DIR, "test.json"))

labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(labels)

# Zostawiamy tylko zwrotki testowe o etykietach znanych modelowi.
test_df = test_df[test_df["label"].isin(label2id)].reset_index(drop=True)

print(f"Liczba klas: {NUM_LABELS}")
print(f"Zbiór testowy: {len(test_df)} zwrotek")
test_df.head(3)

Liczba klas: 20
Zbiór testowy: 1203 zwrotek


,song,section,text,label
0,Marcelo Burlon,refren,Siedzę w Mediolanie jak Marcelo Burlon\nŻycie ...,Żabson
1,All Black,zwrotka,"Zmieniasz się całe życie, umrzesz taki sam, ta...",Quebonafide
2,Hood Angel,refren,"Zarabiam, oh, zarabiam big money, oh\nNie meen...",Żabson


## Wczytanie modelu bazowego i adapterów LoRA

Model bazowy `unsloth/Qwen3.5-4B` ładujemy w bf16 LoRA na GPU, a na sprzęcie bez GPU na CPU bez kwantyzacji.
Adaptery zapisane podczas treningu Unsloth mają inną konwencję nazw parametrów (`model.layers` zamiast `language_model.layers`), więc używamy `load_qwen_peft_model` zamiast zwykłego `PeftModel.from_pretrained`.


In [3]:
def remap_qwen_adapter_key(key: str) -> str:
    """Remap checkpoint keys from Unsloth training format to PeftModel inference format."""
    if "language_model" not in key and ".model.model.layers." in key:
        key = key.replace(
            "base_model.model.model.layers.",
            "base_model.model.model.language_model.layers.",
        )
    if key.endswith(".lora_A.weight"):
        key = key[: -len(".weight")] + ".default.weight"
    elif key.endswith(".lora_B.weight"):
        key = key[: -len(".weight")] + ".default.weight"
    return key


def load_qwen_peft_model(model, adapter_dir: str) -> PeftModel:
    """Załaduj adapter Qwen z remapowaniem kluczy ze starego formatu Unsloth."""
    adapter_path = os.path.join(adapter_dir, "adapter_model.safetensors")
    model = PeftModel.from_pretrained(model, adapter_dir)

    remapped = {}
    with safe_open(adapter_path, framework="pt") as f:
        for key in f.keys():
            if "lora" not in key:
                continue
            remapped[remap_qwen_adapter_key(key)] = f.get_tensor(key)

    params = dict(model.named_parameters())
    loaded = 0
    for key, tensor in remapped.items():
        param = params.get(key)
        if param is None:
            continue
        param.data.copy_(tensor.to(device=param.device, dtype=param.dtype))
        loaded += 1

    print(
        f"Załadowano {loaded}/{len(remapped)} wag LoRA MLP "
        f"(attention LoRA nie było w checkpointcie treningowym)"
    )
    return model

In [4]:
MODEL_NAME = "unsloth/Qwen3.5-4B"
ADAPTER_DIR = "qwen3-5-4b-polish-rap-classifier"
MAX_LENGTH = 650 # ok. 99 percentyl

has_cuda = torch.cuda.is_available()
print(f"Urządzenie: {'GPU (bf16 LoRA)' if has_cuda else 'CPU (bez kwantyzacji)'}")

# Wczytanie modelu bazowego
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=False,
    load_in_16bit=has_cuda,
    dtype=None if has_cuda else "float32",
    auto_model=AutoModelForSequenceClassification,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    trust_remote_code=False,
    ignore_mismatched_sizes=True,
    use_exact_model_name=True,
    device_map="auto",
)

# Nałożenie adapterów LoRA (z remapowaniem kluczy) i głowicy klasyfikacyjnej.
model = load_qwen_peft_model(model, ADAPTER_DIR)
model.eval()

# Tokenizer wczytujemy z katalogu z adapterami.
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if not tokenizer.chat_template:
    tokenizer.chat_template = AutoTokenizer.from_pretrained(MODEL_NAME).chat_template
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

text_config = model.config.get_text_config()
text_config.pad_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Wczytano model i adaptery z: {ADAPTER_DIR}")

Urządzenie: GPU (bf16 LoRA)
==((====))==  Unsloth 2026.6.1: Fast Qwen3_5 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Qwen3_5ForSequenceClassification LOAD REPORT from: unsloth/Qwen3.5-4B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


unsloth/Qwen3.5-4B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


c:\Users\Damia\Desktop\Projekt-PJN\.venv\Lib\site-packages\peft\peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.out_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_qkv.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_z.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_z.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_b.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_b.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.linear_attn.in_proj_a.lora_A.default.weight'

Załadowano 256/256 wag LoRA MLP (attention LoRA nie było w checkpointcie treningowym)
Wczytano model i adaptery z: qwen3-5-4b-polish-rap-classifier


## Formatowanie wejścia i funkcja predykcji

Używamy dokładnie tego samego promptu i formatu wejścia co podczas treningu.


In [5]:
CLASSIFICATION_PROMPT = """Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
{verse}"""


def format_input(verse: str) -> str:
    messages = [{"role": "user", "content": CLASSIFICATION_PROMPT.format(verse=verse)}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        chat_template_kwargs={"enable_thinking": False},
    )


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def predict_artist(verse: str, top_k: int = 5):
    inputs = tokenizer(
        format_input(verse),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)
    probs = model(**inputs).logits.softmax(-1)[0]
    top = probs.topk(min(top_k, NUM_LABELS))
    return [(id2label[i.item()], p.item()) for p, i in zip(top.values, top.indices)]

## Inferencja dla wybranej zwrotki ze zbioru testowego


In [6]:
SAMPLE_INDEX = 0  # Indeks zwrotki ze zbioru testowego

row = test_df.iloc[SAMPLE_INDEX]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")

`use_return_dict` is deprecated! Use `return_dict` instead!


Utwór: Marcelo Burlon
Fragment: Siedzę w Mediolanie jak Marcelo Burlon / Życie nie jest tanie jak Marcelo Burlon / Wiesz że wniosłem ten styl i dlatego mówią
Prawdziwy: Żabson
[TRAFIONO] Predykcja (top-5):
  Żabson                         85.2% <-- prawda
  Louis Villain                  5.1%
  OG Olgierd                     2.1%
  Oki                            1.5%
  Avi                            0.8%


## Inferencja dla losowej zwrotki ze zbioru testowego


In [7]:
row = test_df.sample(1).iloc[0]
verse = row["text"]
true_label = row["label"]

ranking = predict_artist(verse, top_k=5)
top_label = ranking[0][0]
status = "TRAFIONO" if top_label == true_label else "POMYŁKA"

print(f"Utwór: {row.get('song', '—')}")
print(f"Fragment: {' / '.join(verse.splitlines()[:3])}")
print(f"Prawdziwy: {true_label}")
print(f"[{status}] Predykcja (top-5):")
for artist, prob in ranking:
    marker = " <-- prawda" if artist == true_label else ""
    print(f"  {artist:30s} {prob:.1%}{marker}")

Utwór: Eden Hazard
Fragment: W miejscu w którym znikam na noc statystyka to wrogi antonim / Bo zakładając coś w nim najczęściej kończysz goły / Byleby rozbić stoły jak pierdolony Jaruzelski
Prawdziwy: Quebonafide
[TRAFIONO] Predykcja (top-5):
  Quebonafide                    72.3% <-- prawda
  Louis Villain                  7.6%
  OG Olgierd                     5.1%
  Oki                            2.9%
  Guzior                         1.9%
